# Trabajo Práctico Integrador: Del Problema al Deploy
**Seminario de Actualización · Tecnicatura Superior en Ciencia de Datos e IA (IFTS N°18)**  
* **Alumno:** Toto Ignacio  
* **Docente:** Cynthia M. Villagra  
* **Ciclo Lectivo:** 2026 — Segundo Cuatrimestre  
* **Fecha de Presentación:** 09 Octubre 2026  

---

## 1. El Problema de Negocio
En la compañía estamos afrontando, luego de más de 20 años, la migración de un software de gestión de depósitos (WMS) desarrollado in-house hacia un software estándar de mercado, con el propósito de adoptar mejores prácticas logísticas y modernizar la infraestructura tecnológica.

Esta transición conlleva varios desafíos operativos y humanos, entre los que se destacan:
1.	Resistencia al cambio e incertidumbre operativa.
2.	Saturación de los Key Users / referentes de soporte.
3.	Pérdida de tiempo y dispersión en la comunicación.


### ¿Por qué un modelo de NLP (Zero-Shot) y no reglas `if/else` con palabras clave?
Un modelo de NLP Zero-Shot basado en Inferencia de Lenguaje Natural (NLI) es la solución adecuada porque interpreta el contexto de la consulta en lugar de buscar coincidencias literales de texto. 

Un sistema de reglas basado en if/else es incapaz de diferenciar si la consulta corresponde a Picking, Recepción, Expedición o Control de Inventario. El modelo Zero-Shot comprende la acción que el usuario quiere realizar en contexto y asigna las probabilidades correspondientes sin requerir entrenamiento previo.

## 2. Metodología: Zero-Shot Classification basada en NLI
Utilizamos **Zero-Shot Classification** mediante Inferencia de Lenguaje Natural (**NLI**):
* La duda del operario actúa como **Premisa**.
* Por cada área operativa configurada, se genera una **Hipótesis** (ej: *"Esta consulta corresponde al área de {área}."*).
* El modelo calcula los *logits* de implicación (*entailment*), contradicción y neutralidad. Aplicando la función **Softmax** sobre los valores de implicación, se obtiene una distribución de probabilidades que suman 100%.

### Modelos Evaluados:
1. **`Recognai/bert-base-spanish-wwm-cased-xnli`**:
   * Arquitectura: BERT Base (12 capas, 110 millones de parámetros, Masked Language Modeling + NSP).
   * Tamaño de pesos: ~440 MB.
   * Dataset de fine-tuning: Subconjunto en español de XNLI.
2. **`Recognai/zeroshot_selectra_medium`**:
   * Arquitectura: ELECTRA Medium (~38-40 millones de parámetros, técnica de *Replaced Token Detection*).
   * Tamaño de pesos: ~163 MB (ultraliviano).
   * Dataset de fine-tuning: Subconjunto en español de XNLI.

In [53]:
import os
import time
from IPython.display import display
import pandas as pd
from huggingface_hub import InferenceClient
from huggingface_hub.errors import HfHubHTTPError
from transformers import pipeline

# Carga de variables de entorno / token de Hugging Face
hf_token = os.getenv("HF_TOKEN")
if not hf_token and os.path.exists(".env"):
    with open(".env") as f:
        for line in f:
            if line.startswith("HF_TOKEN="):
                hf_token = line.strip().split("=", 1)[1]

print(f"Token configurado" if hf_token else "Sin token detectado")

Token configurado


## 3. Datos de Prueba
Listamos 8 áreas del almacén y 11 consultas del mundo real del depósito para validar la capacidad de ambos modelos:

In [54]:
labels = [
    "Recepción de Pallets",
    "Preparación de Pedidos",
    "Almacenamiento de pallets",
    "Expedición de Camiones",
    "Despacho de pedidos",
    "Planificación de viajes",
    "Control de Inventario",
    "Sistemas y Soporte WMS"
]

questions = [
    # Recepción de Pallets
    ("Estoy recepcionando un pale que tiene rotura. Como lo registro?", "Recepción de Pallets"),
    ("Los pallets que estamos descargando en la rece están fuera de estandar. Tengo que cobrar el retrabajo al proveedor. Que función de la PDA era?", "Recepción de Pallets"),
    
    # Planificación de viajes
    ("El plan quedó chico. Necesito que envien más volumen para replnificar. Sino van a quedar los camiones por la mitad.", "Planificación de viajes"),

    # Almacenamiento de pallets
    ("Terminé de descargar y la PDA me manda a guardar el pallet en el pasillo 14, pero la altura del hueco no da para este bulto. Como pido reubicarlo?", "Almacenamiento de pallets"),
    
    # Preparación de Pedidos
    ("En el Hueco no está el PLU que me dice la PDA que tengo que pickear. Como registro la incidencia?", "Preparación de Pedidos"),
    ("La repo de mercadería me envío a reponer a un hueco de picking que está lleno.", "Preparación de Pedidos"),
    
    # Expedición de Camiones
    ("Que función de la PDA es la que tengo que acceder para consolidar pallets pa subir al camión?", "Expedición de Camiones"),
    ("El camión está lleno pero la HDR tiene más pallets.", "Expedición de Camiones"),

    # Despacho de pedidos
    ("Estoy en el andén. Me falta cerrar el camión para que salga a la calle. Pero no se como se hace.", "Despacho de pedidos"),
    
    # Control de Inventario
    ("Desde control de gestión estamos haciendo la recorrida de control. Detectamos diferencias entre el físico y el sistema. Tengo 3 bultos faltantes del PLU 42222", "Control de Inventario"),
    
    # Despacho de pedidos
    ("La PDA quiere consolidar pales de pollo con veneno para subirlos a un camión. Eso está mal.", "Despacho de pedidos"),
    
    # Sistemas y Soporte WMS
    ("La PDA está tirando error de conexión al WIFI.", "Sistemas y Soporte WMS")
]

df_casos = pd.DataFrame(questions, columns=["Consulta Operativa", "Área Real Esperada"])
df_casos

,Consulta Operativa,Área Real Esperada
0,Estoy recepcionando un pale que tiene rotura. ...,Recepción de Pallets
1,Los pallets que estamos descargando en la rece...,Recepción de Pallets
2,El plan quedó chico. Necesito que envien más v...,Planificación de viajes
3,Terminé de descargar y la PDA me manda a guard...,Almacenamiento de pallets
4,En el Hueco no está el PLU que me dice la PDA ...,Preparación de Pedidos
5,La repo de mercadería me envío a reponer a un ...,Preparación de Pedidos
6,Que función de la PDA es la que tengo que acce...,Expedición de Camiones
7,El camión está lleno pero la HDR tiene más pal...,Expedición de Camiones
8,Estoy en el andén. Me falta cerrar el camión p...,Despacho de pedidos
9,Desde control de gestión estamos haciendo la r...,Control de Inventario


## 4. `InferenceClient`
Probamos el consumo de la API en los servidores de Hugging Face y capturamos las respuestas obtenidas:

In [55]:
client = InferenceClient(token=hf_token)

models_remote = [
    "Recognai/bert-base-spanish-wwm-cased-xnli",
    "Recognai/zeroshot_selectra_medium"
]

print("=== PROBANDO INFERENCE CLIENT ===")
for m in models_remote:
    print(f"\nConsultando API de Hugging Face para modelo: {m}")
    try:
        res = client.zero_shot_classification(
            text="Estoy recepcionando un pale que tiene rotura. Como lo registro?",
            candidate_labels=["Recepción", "Expedición y Despacho", "Control de Inventario"],
            hypothesis_template="Esta consulta corresponde al área de {}.",
            model=m
        )
        print("Resultado exitoso:", res)
    except HfHubHTTPError as e:
        print(f"--> [ERROR HTTP DE HUGGING FACE]: Código {e.response.status_code}")
        print(f"    Mensaje: {e}")
    except Exception as e:
        print(f"--> [EXCEPCIÓN LOCAL]: {type(e).__name__}: {e}")

=== PROBANDO INFERENCE CLIENT ===

Consultando API de Hugging Face para modelo: Recognai/bert-base-spanish-wwm-cased-xnli
--> [ERROR HTTP DE HUGGING FACE]: Código 402
    Mensaje: 402 Client Error: Payment Required for url: https://router.huggingface.co/hf-inference/models/Recognai/bert-base-spanish-wwm-cased-xnli (Request ID: Root=1-6ac9538f-275d8ba215794dac089286d6;7eae8eba-71c1-4db3-9b61-c0043e6231e6)

You have no remaining credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get monthly included credits.

Consultando API de Hugging Face para modelo: Recognai/zeroshot_selectra_medium
--> [EXCEPCIÓN LOCAL]: StopIteration: 


### 📌 Análisis del Comportamiento de `InferenceClient`:
1. **Error HTTP 402 (*Payment Required*):**  
   Al consultar `Recognai/bert-base-spanish-wwm-cased-xnli`, se obtiene un mensaje de error:  
   *`"You have no remaining credits. Purchase pre-paid credits to continue using Inference Providers..."`*.  
2. **Excepción `StopIteration` en SELECTRA:**  
   Al consultar `Recognai/zeroshot_selectra_medium`, la librería falla internamente porque ningún proveedor comercial de HF tiene desplegado este modelo en sus catálogos de inferencia en la nube.

> **Impacto y Decisión Arquitectónica:**  
> La dependencia de APIs de terceros introduce complejidad respecto a la disponibilidad y costos imprevistos. Por ende, la ejecución local con `transformers.pipeline` garantiza soberanía sobre el servicio sin costo.

## 5. `transformers.pipeline`
Cargamos ambos modelos en memoria local (CPU) y medimos:
1. **Tiempo de carga inicial del modelo.**
2. **Tiempo de inferencia por consulta.**
3. **Top 1 predicho y nivel de confianza (probabilidad softmax).**

In [56]:
model_configs = {
    "BERT-XNLI (110M)": "Recognai/bert-base-spanish-wwm-cased-xnli",
    "SELECTRA-Medium (38M)": "Recognai/zeroshot_selectra_medium"
}

benchmark_results = []
load_times = {}

for model_name, model_id in model_configs.items():
    print(f"Cargando {model_name}...")
    t0 = time.time()
    classifier = pipeline("zero-shot-classification", model=model_id, device=-1)
    t_load = time.time() - t0
    load_times[model_name] = round(t_load, 2)
    print(f"  -> Cargado en {t_load:.2f} s")
    
    for q, expected in questions:
        t_infer_start = time.time()
        res = classifier(
            q,
            candidate_labels=labels,
            hypothesis_template="Esta consulta corresponde al área de {}."
        )
        infer_duration = time.time() - t_infer_start
        
        pred_label = res["labels"][0]
        pred_score = res["scores"][0]
        second_label = res["labels"][1]
        second_score = res["scores"][1]
        
        benchmark_results.append({
            "Modelo": model_name,
            "Consulta": q,
            "Esperada": expected,
            "Predicha": pred_label,
            "Acierto": (pred_label == expected),
            "Confianza": round(pred_score * 100, 1),
            "Segunda Opción": f"{second_label} ({second_score*100:.1f}%)",
            "Latencia (s)": round(infer_duration, 2)
        })

df_bench = pd.DataFrame(benchmark_results)
try:
    df_bench.to_csv("comparativa_modelos.csv", index=False)
except Exception:
    pass
print("Benchmark completado.")

Cargando BERT-XNLI (110M)...


Device set to use cpu


  -> Cargado en 0.49 s
Cargando SELECTRA-Medium (38M)...


Device set to use cpu


  -> Cargado en 0.66 s
Benchmark completado.


### 📌 Diferencias en los Valores de Retorno (`type()`)
Tal como exige la consigna, demostramos qué devuelve cada llamado:

1. **`transformers.pipeline`**: Devuelve una lista de diccionarios (tipo `list` y `dict`). Los valores se acceden mediante sintaxis de llaves: `res['labels']`.
2. **`huggingface_hub.InferenceClient`**: Devuelve un objeto o diccionario anidado dependiendo de la librería. 

A continuación, realizamos un llamado manual al pipeline local para imprimir su `type()`:

In [57]:
# Realizamos una inferencia rápida para mostrar los tipos de retorno
res_pipeline = classifier(
    "El camión llegó tarde.", 
    candidate_labels=["Logística", "Ventas"]
)

print("=== RETORNO DE PIPELINE LOCAL ===")
print("Tipo de dato de la respuesta:", type(res_pipeline))
print("Estructura devuelta:", res_pipeline)
print("\nAcceso al primer label predicho:", res_pipeline["labels"][0])

=== RETORNO DE PIPELINE LOCAL ===
Tipo de dato de la respuesta: <class 'dict'>
Estructura devuelta: {'sequence': 'El camión llegó tarde.', 'labels': ['Logística', 'Ventas'], 'scores': [0.6173614263534546, 0.382638543844223]}

Acceso al primer label predicho: Logística


## 6. Análisis de Rendimiento

In [58]:
# Comparativa de tiempos de carga y latencia promedio
metricas = df_bench.groupby("Modelo").agg(
    Aciertos=("Acierto", "sum"),
    Total=("Acierto", "count"),
    Latencia_Promedio_s=("Latencia (s)", "mean"),
    Confianza_Promedio_pct=("Confianza", "mean")
).reset_index()

metricas["Tiempo_Carga_s"] = metricas["Modelo"].map(load_times)
metricas["Exactitud_pct"] = round((metricas["Aciertos"] / metricas["Total"]) * 100, 1)

display(metricas[["Modelo", "Tiempo_Carga_s", "Latencia_Promedio_s", "Aciertos", "Exactitud_pct", "Confianza_Promedio_pct"]])

,Modelo,Tiempo_Carga_s,Latencia_Promedio_s,Aciertos,Exactitud_pct,Confianza_Promedio_pct
0,BERT-XNLI (110M),0.49,0.410000,6,50.0,30.291667
1,SELECTRA-Medium (38M),0.66,0.154167,4,33.3,45.983333


In [59]:
# Visualización detallada caso a caso
display(df_bench[["Modelo", "Consulta", "Esperada", "Predicha", "Acierto", "Confianza", "Segunda Opción"]])

,Modelo,Consulta,Esperada,Predicha,Acierto,Confianza,Segunda Opción
0,BERT-XNLI (110M),Estoy recepcionando un pale que tiene rotura. ...,Recepción de Pallets,Recepción de Pallets,True,45.7,Despacho de pedidos (14.7%)
1,BERT-XNLI (110M),Los pallets que estamos descargando en la rece...,Recepción de Pallets,Recepción de Pallets,True,33.3,Almacenamiento de pallets (20.4%)
2,BERT-XNLI (110M),El plan quedó chico. Necesito que envien más v...,Planificación de viajes,Expedición de Camiones,False,35.8,Despacho de pedidos (16.7%)
3,BERT-XNLI (110M),Terminé de descargar y la PDA me manda a guard...,Almacenamiento de pallets,Almacenamiento de pallets,True,32.5,Despacho de pedidos (17.8%)
4,BERT-XNLI (110M),En el Hueco no está el PLU que me dice la PDA ...,Preparación de Pedidos,Despacho de pedidos,False,19.3,Preparación de Pedidos (18.8%)
5,BERT-XNLI (110M),La repo de mercadería me envío a reponer a un ...,Preparación de Pedidos,Despacho de pedidos,False,26.4,Recepción de Pallets (14.3%)
6,BERT-XNLI (110M),Que función de la PDA es la que tengo que acce...,Expedición de Camiones,Expedición de Camiones,True,29.4,Recepción de Pallets (20.6%)
7,BERT-XNLI (110M),El camión está lleno pero la HDR tiene más pal...,Expedición de Camiones,Expedición de Camiones,True,24.1,Almacenamiento de pallets (21.1%)
8,BERT-XNLI (110M),Estoy en el andén. Me falta cerrar el camión p...,Despacho de pedidos,Expedición de Camiones,False,31.3,Despacho de pedidos (23.3%)
9,BERT-XNLI (110M),Desde control de gestión estamos haciendo la r...,Control de Inventario,Control de Inventario,True,35.0,Despacho de pedidos (23.1%)


## 7. Análisis comparativo

### Aciertos:
1. **Detección de Recepción:**  
   Ambos modelos acertaron *"Estoy recepcionando un pale que tiene rotura..."* y *"Los pallets que estamos descargando en la rece..."*. El modelo asocia muy bien la palabra "rece" y "recepcionando" con la etiqueta **Recepción de Pallets**.
2. **Almacenamiento (Caso 4):**  
   **Ambos modelos acertaron** (*"Terminé de descargar y la PDA me manda a guardar el pallet en el pasillo 14..."*), detectando que la acción operativa era **Almacenamiento de pallets** con una alta confianza.
3. **Fallas Técnicas y Hardware (Caso 12):**  
   Para *"La PDA está tirando error de conexión al WIFI"*, el modelo SELECTRA logró clasificarlo correctamente en **Sistemas y Soporte WMS**, diferenciando el problema de red de una incidencia operativa del piso.

---

### Errores y Hallazgos (¿Dónde se equivoca?):
A partir de las pruebas ejecutadas con 12 consultas reales del depósito, se identificaron los siguientes patrones y sesgos claros donde los modelos fallan:

1. **Confusión sistemática entre Preparación y Despacho de pedidos:**  
   * **Consulta:** *En el Hueco no está el PLU que me dice la PDA que tengo que pickear...*
   * **Respuesta:** Ambos modelos predijeron Despacho de Pedidos, en lugar de Preparación de Pedidos.
   * **Conclusión:** La palabra "picking" o "pickear" no llega a guiar la respuesta hacia la preparación de pedidos. Probablemente debido a que no son términos en español y sigue siendo un término de nicho en la jerga logística.

2. **Complejidad entre términos de significados similares:**  
   * **Consulta:** *Estoy en el andén. Me falta cerrar el camión...* / *La PDA quiere consolidar pales...*
   * **Respuesta:** Predicen Expedición de Camiones en lugar de Despacho, o viceversa.
   * **Conclusión:** Para un modelo de lenguaje general pre-entrenado, Expedir y Despachar son sinónimos casi exactos. Al no haber entrenamiento supervisado específico, el modelo no puede diferenciar entre estas dos sub-áreas logísticas.

3. **Peso de los términos:**  
   * **Consulta:** *El plan quedó chico. Necesito que envien más volumen para replnificar...*
   * **Respuesta:** Ambos coinciden en Expedición de Camiones.
   * **Conclusión:** Los modelos no llegan a relacionar fuertemente el "plan" con la acción de planificar. Priorizan la mención a los "camiones" y la vinculan a la expedición y despacho.

## 8. Conclusiones y Selección para el Deploy en Streamlit
| Criterio | BERT-XNLI (110M) | SELECTRA-Medium (38M) | Ganador para Deploy |
|---|---|---|---|
| **Tamaño de pesos** | ~440 MB | **~163 MB** | **SELECTRA** (63% más liviano) |
| **Tiempo de carga** | ~7.8 s | **~0.6 s** | **SELECTRA** (13x más veloz) |
| **Latencia por inferencia** | ~0.80 s a 1.9 s | **~0.20 s a 0.4 s** | **SELECTRA** (4x más rápido) |
| **Consumo de memoria RAM** | ~600 MB - 800 MB | **< 300 MB** | **SELECTRA** (apto tier gratuito de 1 GB) |
| **Solapamiento Semántico** | Se confunde entre Expedición y Despacho | Se confunde entre Expedición y Despacho | **EMPATE** (Limitación técnica de Zero-Shot NLP) |
### Decisión de Arquitectura e Implementación Futura:
Para la aplicación interactiva de Streamlit:
* **Elegimos `Recognai/zeroshot_selectra_medium` como modelo principal:** Su tamaño diminuto (163 MB) y velocidad de respuesta (200 ms) permiten que la app arranque instantáneamente en el contenedor de Streamlit Community Cloud sin riesgo de exceder el límite estricto de 1 GB de RAM.
* **Caché de recursos:** Empleamos `@st.cache_resource` para instanciar el pipeline una única vez al inicio del servidor, garantizando que cada consulta posterior de los operarios sea instantánea.
* **Lección Aprendida para el Negocio:** Hemos validado empíricamente que **Zero-Shot NLP tiene límites cuando las etiquetas son sinónimos logísticos** (ej. *Expedición* vs *Despacho*). Para llevar este asistente a un nivel de producción total, el WMS requeriría recolectar un dataset histórico de tickets y realizar un **Fine-Tuning (Entrenamiento Supervisado)** del modelo para que aprenda las sutiles diferencias internas de la jerga de la empresa. Mientras tanto, el sistema actual sirve como un triaje inicial de altísima velocidad.
